In [ ]:
import pandas as pd
import numpy as np
import sympy as sp
import math
import scipy
import matplotlib.pyplot as plt
import scikit_posthocs as sp
import statsmodels.api as sm
from statsmodels.stats.anova import AnovaRM
import pingouin as pg

In [ ]:
df_original = pd.read_csv('combine_pilot.csv')
display(df_original)

In [ ]:
# filter only fov = 90, 110, and 130
df = df_original[(df_original['fov'] == 90) | (df_original['fov'] == 110) | (df_original['fov'] == 130)]

# Major Comparison

In [ ]:
fov_list = [90, 110, 130]

for i, k in enumerate(df.columns):
  if (i < 2): continue
  print("\n┉┉┉┉┉┉┉┉┉┉┉┉\n🔷Column: ", k)
  f90 = df[(df['fov'] == 90)][k].to_numpy()
  f110 = df[(df['fov'] == 110)][k].to_numpy()
  f130 = df[(df['fov'] == 130)][k].to_numpy()

  # drop nan and 0
  f90 = f90[~np.isnan(f90)]
  f90 = f90[f90 != 0]
  f110 = f110[~np.isnan(f110)]
  f110 = f110[f110 != 0]
  f130 = f130[~np.isnan(f130)]
  f130 = f130[f130 != 0]

  compare_list = []
  compares = []
  if (len(f90) != 0):
    compares.append(f90)
    compare_list.append("f90")
  if (len(f110) != 0):
    compares.append(f110)
    compare_list.append("f110")
  if (len(f130) != 0):
    compares.append(f130)
    compare_list.append("f130")

  print("\n[Comparing]\n0:", compare_list[0], "\n1:", compare_list[1], "\n2:", compare_list[2])
  print("\n++ Shapiro & Levene ++")
  p_shap = [0, 0, 0]
  for q, r in enumerate(compares):
    p_shap[q] = scipy.stats.shapiro(r)[1]
  p_eqv = scipy.stats.levene(compares[0], compares[1], compares[2])[1]
  print("Shapiro:", p_shap)
  print("Levene:", p_eqv)

  if p_eqv > 0.05 and all(p > 0.05 for p in p_shap):
    print("⫸ Parametric")

    # print("++ one-way ANOVA ++")
    # stat = scipy.stats.f_oneway(compares[0], compares[1], compares[2])
    # p_value = stat.pvalue
    # print(stat)

    print("++ repeated measurements one-way ANOVA ++")
    df_anv = df[['user', 'fov', k]].copy()
    df_anv.columns = ['Subject', 'Condition', 'Measurement']
    # display(df_anv)

    # obs_counts = df_anv.groupby(['Subject', 'Condition']).size()
    # print(obs_counts)

    print("++ Mauchly Spherical Test ++")
    mauchly_test = pg.sphericity(data=df_anv, dv='Measurement', within='Condition', subject='Subject')
    print("Mauchly Test:", mauchly_test)
    if (mauchly_test[4] > 0.05):
      print("⏺️ Sphericity passed")
      anova_rm = AnovaRM(df_anv, 'Measurement', 'Subject', within=['Condition'])
      result = anova_rm.fit()
      # print(result.summary())
      anova_table = result.anova_table
      # print(anova_table)

      F_statistic = anova_table['F Value'][0]
      p_value = anova_table['Pr > F'][0]
      print(f"F-statistic: {F_statistic}, P-value: {p_value}")
    else:
      print("⛔️ Sphericity failed")
      aov = pg.rm_anova(data=df_anv, dv='Measurement', within='Condition', subject='Subject', detailed=True, correction=True)
      print(aov)

    if (p_value < 0.05):
      print("✅ they have significant differences with p-value", p_value)
      print("++ Tukey's HSD ++")
      stat = scipy.stats.tukey_hsd(compares[0], compares[1], compares[2])
      print(stat)
      for q, r in enumerate(stat.pvalue):
        for s, t in enumerate(r):
          if (t < 0.05):
            print('✅', compare_list[q], "and", compare_list[s], "have significant differences with p-value", t)
  else:
    print("⫷ Non-parametric")
    print("++ Kruskal-Wallis ++")
    stat = scipy.stats.kruskal(compares[0], compares[1], compares[2])
    print(stat)
    if (stat.pvalue < 0.05):
      print("✅ they have significant differences with p-value", stat.pvalue)
      print("++ Dunn's with Bonferroni ++")
      p_values = sp.posthoc_dunn(compares, p_adjust='bonferroni')
      print(p_values)
      for q in range(0, p_values.shape[0]-1):
          for r in range(0, p_values.shape[1]-1):
              if p_values.iloc[q, r] < 0.05:
                  print('✅', compare_list[q], "and", compare_list[r], "have significant differences with p-value", t)
      

# Pairwise Comparison

In [ ]:
# t-test / wilcoxon for specific items (greedy)

com_set = [
  ['head_qu_distance', 90, 110],
  ['head_qu_distance', 90, 130],
  ['head_qu_distance', 110, 130],
  
  ['head_qu_mean_velocity', 90, 110],
  ['head_qu_mean_velocity', 90, 130],
  ['head_qu_mean_velocity', 110, 130],

  ['head_qu_mean_acc', 90, 110],
  ['head_qu_mean_acc', 90, 130],
  ['head_qu_mean_acc', 110, 130],

  ['rightHand_qu_distance', 90, 110],
  ['rightHand_qu_distance', 90, 130],
  ['rightHand_qu_distance', 110, 130],

  ['leftHand_qu_distance', 90, 110],
  ['leftHand_qu_distance', 90, 130],
  ['leftHand_qu_distance', 110, 130],

  ['hands_mean_distance', 90, 110],
  ['hands_mean_distance', 90, 130],
  ['hands_mean_distance', 110, 130],
]

for i, k in enumerate(com_set):
  print("///", k[0] ,"///")
  
  a = df[df['fov'] == k[1]][k[0]]
  b = df[df['fov'] == k[2]][k[0]]

  print(k, 'Shapiro-Wilk Normality Test')
  stat1, p1 = scipy.stats.shapiro(a)
  print('Statistics=%.4f, p=%.4f' % (stat1, p1))
  stat2, p2 = scipy.stats.shapiro(b)
  print('Statistics=%.4f, p=%.4f' % (stat2, p2))

  print('Levene Test for Equal Variance')
  stat_eqv, p_eqv = scipy.stats.levene(a, b)
  print('Statistics=%.4f, p=%.4f' % (stat_eqv, p_eqv))
  
  if (p1 > 0.05 and p2 > 0.05):
    print('\n⫸ Parametric')
    print('\n- paired -')
    if (a.shape[0] == b.shape[0]):
      stat, p= scipy.stats.ttest_rel(
                                  a = a, 
                                  b = b,
                                  alternative='less'
                                )
      print("\nt-test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[1], 'is less than', k[2], 'in terms of', k[0])
      else:
        print(k[1], 'is not less than', k[2], 'in terms of', k[0])
      print("-◎-")
      stat, p= scipy.stats.ttest_rel(
                                  a = b, 
                                  b = a,
                                  alternative='less'
                                )
      print("t-test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[2], 'is less than', k[1], 'in terms of', k[0])
      else:
        print(k[2], 'is not less than', k[1], 'in terms of', k[0])
      print("\n--◎--◎--◎--")
      stat, p= scipy.stats.ttest_rel(
                                  a = a, 
                                  b = b,
                                  alternative='greater'
                                )
      print("\nt-test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[1], 'is greater than', k[2], 'in terms of', k[0])
      else:
        print(k[1], 'is not greater than', k[2], 'in terms of', k[0])
      print("-◎-")
      stat, p= scipy.stats.ttest_rel(
                                  a = b, 
                                  b = a,
                                  alternative='greater'
                                )
      print("t-test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[2], 'is greater than', k[1], 'in terms of', k[0])
      else:
        print(k[2], 'is not greater than', k[1], 'in terms of', k[0])
      print("\n--◎--◎--◎--")
      stat, p= scipy.stats.ttest_rel(
                                  a = a, 
                                  b = b,
                                  alternative='two-sided'
                                )
      print("\nt-test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[1], 'is different to', k[2], 'in terms of', k[0])
      else:
        print(k[1], 'is not different to', k[2], 'in terms of', k[0])
    else:
      print("🚫The size of samples are not equal")
  else:
    print('\n⫷ Non-parametric')
    print('\n- paired -')
    if (a.shape[0] == b.shape[0]):
      stat, p= scipy.stats.wilcoxon(
                                  x = a, 
                                  y = b, 
                                  alternative='less'
                                )
      print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[1], 'is less than', k[2], 'in terms of', k[0])
      else:
        print(k[1], 'is not less than', k[2], 'in terms of', k[0])
      print("-◎-")
      stat, p= scipy.stats.wilcoxon(
                                  x = b, 
                                  y = a, 
                                  alternative='less'
                                )
      print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[2], 'is less than', k[1], 'in terms of', k[0])
      else:
        print(k[2], 'is not less than', k[1], 'in terms of', k[0])
      print("\n--◎--◎--◎--")
      stat, p= scipy.stats.wilcoxon(
                                  x = a, 
                                  y = b, 
                                  alternative='greater'
                                )
      print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[1], 'is greater than', k[2], 'in terms of', k[0])
      else:
        print(k[1], 'is not greater than', k[2], 'in terms of', k[0])
      print("-◎-")
      stat, p= scipy.stats.wilcoxon(
                                  x = b, 
                                  y = a, 
                                  alternative='greater'
                                )
      print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[2], 'is greater than', k[1], 'in terms of', k[0])
      else:
        print(k[2], 'is not greater than', k[1], 'in terms of', k[0])
      print("\n--◎--◎--◎--")
      stat, p= scipy.stats.wilcoxon(
                                  x = a, 
                                  y = b, 
                                  alternative='two-sided'
                                )
      print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[1], 'is different to', k[2], 'in terms of', k[0])
      else:
        print(k[1], 'is not different to', k[2], 'in terms of', k[0])
    else:
      print("🚫The size of samples are not equal")
  print("\n\n")

# Basic Info

In [ ]:
f130_hq_mean = df[(df['fov'] == 130)]['head_qu_distance'].mean()
f110_hq_mean = df[(df['fov'] == 110)]['head_qu_distance'].mean()
f90_hq_mean = df[(df['fov'] == 90)]['head_qu_distance'].mean()

print("f130_hq_mean:", f130_hq_mean)
print("f110_hq_mean:", f110_hq_mean)
print("f90_hq_mean:", f90_hq_mean)

diff_130_90 = (f130_hq_mean - f90_hq_mean) / f90_hq_mean
diff_130_110 = (f130_hq_mean - f110_hq_mean) / f110_hq_mean

print("diff_130_90:", diff_130_90)
print("diff_130_110:", diff_130_110)

# Plot

In [ ]:
# plot about head_qu_distance
plt.figure(figsize=(6, 6))
_ = plt.boxplot([df[(df['fov'] == 90)]['head_qu_distance'], df[(df['fov'] == 110)]['head_qu_distance'], df[(df['fov'] == 130)]['head_qu_distance']], widths=0.4)
plt.xticks([1, 2, 3], ['90°', '110°', '130°'])
plt.ylabel('Quaternion Distance of Head Displacement')
# title
plt.title('Quaternion Distance of Head Displacement in Different FOVs')

In [ ]:
# plot about head_qu_distance ✅
data_to_plot = [
    df[(df['fov'] == 90) & (df['head_qu_distance'] != 0)]['head_qu_distance'].dropna(),
    df[(df['fov'] == 110) & (df['head_qu_distance'] != 0)]['head_qu_distance'].dropna(),
    df[(df['fov'] == 130) & (df['head_qu_distance'] != 0)]['head_qu_distance'].dropna(),
    df_original[(df_original['fov'] == 150) & (df_original['head_qu_distance'] != 0)]['head_qu_distance'].dropna(),
    df_original[(df_original['fov'] == 170) & (df_original['head_qu_distance'] != 0)]['head_qu_distance'].dropna()
]

_ = plt.boxplot(data_to_plot)
plt.xticks([1, 2, 3, 4, 5], ['90°', '110°', '130°', '150°', '170°'])
plt.ylabel('Quaternion Distance of Head Displacement')
# title
plt.title('Quaternion Distance of Head Displacement in Different FOVs')

In [ ]:
list_f150 = [
    "P021", "P007", "P001", "P022", "P023", "P031", "P024"
  ]
list_f170 = [
  "P023", "P031", "P024"
]

# filter
df_f150 = df_original[df_original['fov'] == 150]
df_f150 = df_f150[df_f150['user'].isin(list_f150)]
df_f170 = df_original[df_original['fov'] == 170]
df_f170 = df_f170[df_f170['user'].isin(list_f170)]

In [ ]:
plt.figure(figsize=(4, 4))
plt.rcParams['figure.dpi'] = 300

data_to_plot = [
    df[(df['fov'] == 90) & (df['head_qu_distance'] != 0)]['head_qu_distance'].dropna(),
    df[(df['fov'] == 110) & (df['head_qu_distance'] != 0)]['head_qu_distance'].dropna(),
    df[(df['fov'] == 130) & (df['head_qu_distance'] != 0)]['head_qu_distance'].dropna(),
    df_f150['head_qu_distance'].dropna(),
    df_f170['head_qu_distance'].dropna()
]

# Define FOV categories for x-axis labels and a small offset to separate points within each category
fovs = ['90°', '110°', '130°', '150°', '170°']
offsets = np.linspace(-0, 0, len(data_to_plot))

# Plot each group with a different color and apply offsets to distinguish them on the x-axis
for i, (data, offset) in enumerate(zip(data_to_plot, offsets)):
    x = np.full(len(data), i + 1) + offset  # Apply offset to each group to spread them out along the x-axis
    plt.scatter(x, data, alpha=0.7, label=f'FOV {fovs[i]}')

# Set the ticks and labels on the x-axis to correspond to your different FOV groups
plt.xticks(range(1, len(fovs) + 1), fovs)
plt.ylabel('Quaternion Distance of Head Displacement')
plt.title('Quaternion Distance of Head Displacement in Different FOVs')
# plt.legend()

# plt.style.use('dark_background')
plt.style.use("classic")

plt.show()